# Data exploration
Needs the tables from step 1 and 2.

In [ ]:
import sys
sys.path.append("..")  # settings.py is one folder up

import matplotlib.pyplot as plt
import settings

sql = settings.query_to_df

### New kidney patients per year

In [ ]:
# first kidney diagnosis per year (first_kidney_dx is made in 2_cohort.sql)
new_per_year = sql("""
    SELECT EXTRACT(YEAR FROM first_dx_date)::int AS year, COUNT(*) AS new_patients
    FROM first_kidney_dx
    WHERE first_dx_date >= '2015-01-01'
    GROUP BY year ORDER BY year
""")
new_per_year.plot(x="year", y="new_patients", kind="bar", legend=False, title="New kidney patients per year")
plt.show()

### Class balance

In [ ]:
features = sql("SELECT * FROM features")

print(features["label"].value_counts())
print("Share of positives: {:.2%}".format(features["label"].mean()))

~0.5% positives -> accuracy is useless here (all 0 = 99.5% accurate). Using PR-AUC.

### Label 0 vs label 1

In [ ]:
# mean per group. for 0/1 columns the mean = share with that condition
columns = ["age", "has_hypertension", "has_diabetes", "last_egfr", "last_creatinine",
           "last_systolic_bp", "last_bmi", "n_visits"]
features.groupby("label")[columns].mean().T.round(2)

### Missing labs (% per group)

In [ ]:
# missing = test not done in the 2 years before the snapshot
labs = ["last_egfr", "last_creatinine", "last_hba1c", "last_albumin_creatinine_ratio", "last_systolic_bp"]
features.groupby("label")[labs].apply(lambda df: df.isna().mean() * 100).T.round(1)

### Notes
- positives ~0.5% per year, very imbalanced
- future kidney patients: older, much more hypertension (86% vs 18%), more diabetes
- eGFR is NOT lower on average for label 1. most new diagnoses are stage 1-2, where eGFR can still be normal. so no single lab separates the groups
- label 1 patients have far fewer missing labs (they get tested more) -> keep a "missing" indicator in the model